In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

# Графикийн дизайныг цэвэрхэн болгох
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Датагаа ачааллах
df = pd.read_csv("features_df.csv")

# График 1: Гомдлын түвшнээр цуцлалтын хувь
df.groupby("complaint_level")["is_churned"].mean().plot.bar(
    ax=axes[0, 0], color="#e74c3c"
)
axes[0, 0].set_title("1. Churn Rate by Complaint Level", fontsize=12, fontweight="bold")
axes[0, 0].set_ylabel("Churn Share")
axes[0, 0].set_xlabel("Complaint Level")
axes[0, 0].tick_params(axis="x", rotation=0)

# График 2: Ашигласан хугацаагаар (Tenure) цуцлалтын хувь
df.groupby("tenure_bucket")["is_churned"].mean().plot.bar(
    ax=axes[0, 1], color="#3498db"
)
axes[0, 1].set_title("2. Churn Rate by Tenure (Months)", fontsize=12, fontweight="bold")
axes[0, 1].set_ylabel("Churn Share")
axes[0, 1].set_xlabel("Tenure Bucket")
axes[0, 1].tick_params(axis="x", rotation=30)

# График 3: Сүүлд төлбөр төлсөн хугацаагаар цуцлалтын хувь
df.groupby("payment_recency_bucket")["is_churned"].mean().plot.bar(
    ax=axes[1, 0], color="#f39c12"
)
axes[1, 0].set_title("3. Churn Rate by Days Since Last Payment", fontsize=12, fontweight="bold")
axes[1, 0].set_ylabel("Churn Share")
axes[1, 0].set_xlabel("Days Since Payment")
axes[1, 0].tick_params(axis="x", rotation=30)

# 4. K-Means Хэрэглэгчийн сегментчлэл (Clustering)
seg_cols = ["monthly_usage_gb", "monthly_spend", "tenure_months"]
seg_df = df.dropna(subset=seg_cols).copy()
Xs = StandardScaler().fit_transform(seg_df[seg_cols])

# 3 сегментэд хуваах
seg_df["segment"] = KMeans(n_clusters=3, n_init=10, random_state=42).fit_predict(Xs)

# График 4: Сегментүүдийн хэрэглээ ба төлбөрийн хамаарал
sns.scatterplot(
    data=seg_df, x="monthly_usage_gb", y="monthly_spend",
    hue="segment", palette="Set1", alpha=0.8, ax=axes[1, 1]
)
axes[1, 1].set_title("4. Customer Segments (Usage vs Spend)", fontsize=12, fontweight="bold")
axes[1, 1].set_xlabel("Monthly Usage (GB)")
axes[1, 1].set_ylabel("Monthly Spend ($)")

plt.tight_layout()
plt.show()

# Сегмент тус бүрийн цуцлалтын хувийг шалгах
print("--- СЕГМЕНТ БҮРИЙН ЦУЦЛАЛТЫН ХУВЬ ---")
segment_summary = seg_df.groupby("segment").agg(
    churn_rate=("is_churned", "mean"),
    avg_usage=("monthly_usage_gb", "mean"),
    avg_spend=("monthly_spend", "mean"),
    avg_tenure=("tenure_months", "mean"),
    count=("is_churned", "count")
)
print(segment_summary)